In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import sys
sys.path.append("../aws_utils")
import aws_utils

#only one should be open
env = aws_utils.get_env_prefix()
#env = 'p'#for manual run - WE RUN WITH THIS UNTIL WE MOVE TABLES TO PROD
#env = aws_utils.get_env_prefix() # for auto run

In [0]:
# Check if EXPORT_BUSINESS is empty
result_business = spark.sql(f"SELECT 1 FROM `{env}-cdh-delivery`.tiresias.EXPORT_BUSINESS LIMIT 1").collect()

# Check if EXPORT_CONSUMER is empty
result_consumer = spark.sql(f"SELECT 1 FROM `{env}-cdh-delivery`.tiresias.EXPORT_CONSUMER LIMIT 1").collect()

# Check if *both* are empty
if not result_business and not result_consumer:
    print("Both EXPORT_BUSINESS and EXPORT_CONSUMER tables are empty. No need for backup, skipping history notebook run.")
    dbutils.notebook.exit("Skipping: Both export tables are empty , no need for history notebook run.")
else:
    print("One or both export tables contain data. Continuing execution.")

## Business Table

In [0]:
date1 = spark.sql(f"SELECT DISTINCT rpt_date FROM `{env}-cdh-delivery`.tiresias.EXPORT_BUSINESS ").collect()[0][0]

date2 = spark.sql(f"select MAX(rpt_date)  from `{env}-cdh-delivery`.tiresias.export_business_hist").collect()[0][0]

# Check if they are equal
if date1 == date2:
    raise ValueError(f"Error: There are already data in business_hist for: {date1}. Stopping execution.")
else:
    print(f"Dates are different: {date1} vs {date2}, continuing execution.")

In [0]:
export_business_hist_insert = spark.sql(f"""
INSERT INTO `{env}-cdh-delivery`.tiresias.export_business_hist (
    rpt_date,
    created,
    portfolio,
    portfolioid,
    `ΑΥΞΩΝ_ΑΡΙΘΜΟΣ`,
    `ΚΩΔΙΚΟΣ_ΚΑΤΑΣΤΗΜΑΤΟΣ`,
    `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`,
    `ΗΜ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`,
    `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΟΥ_ΟΡΙΟΥ_ΠΟΣΟΥ_ΕΓΚΡΙΣΗΣ_ΜΕ_ΑΛΛΗ_ΕΠΙΧΕΙΡΗΣΗ`,
    `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΕΓΚΡΙΣΗΣ_ΧΟΡΗΓΗΣΕΩΝ_ΤΗΣ_ΙΔΙΑΣ_ΕΠΙΧΕΙΡΗΣΗΣ`,
    `ΑΡ_ΔΟΣΕΩΝ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΥΠΟΛΕΙΠΟΜΕΝΕΣ_ΔΟΣΕΙΣ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΔΕΙΚΤΗΣ_ΚΑΘΥΣΤΕΡΗΣΗΣ`,
    `ΠΟΣΟ_ΔΟΣΗΣ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΠΟΣΟ_ΕΠΟΜΕΝΗΣ_ΕΛΑΧΙΣΤΗΣ_ΚΑΤΑΒΟΛΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`,
    `ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΟΡΙΟ_ΧΟΡΗΓΗΣΗΣ`,
    `ΣΥΧΝΟΤΗΤΑ_ΔΟΣΕΩΝ_Η_ΕΛΑΧΙΣΤΩΝ_ΚΑΤΑΒΟΛΩΝ`,
    `ΠΟΣΟ_ΠΛΗΡΩΜΩΝ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΜΗΝΑ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΚΙΝΗΣΗΣ_ΛΟΓΑΡΙΑΣΜΟΥ`,
    `ΕΝΗΜΕΡΟ_ΥΠΟΛΟΙΠΟ`,
    `ΥΠΟΛΟΙΠΟ_ΣΕ_ΚΑΘΥΣΤΕΡΗΣΗ`,
    `ΑΡΝΗΤΙΚΟ_ΥΠΟΛΟΙΠΟ`,
    `ΣΥΝΟΛΟ_ΟΦΕΙΛΗΣ`,
    `ΗΜ_ΕΓΚΡΙΣΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,
    `ΔΕΙΚΤΗΣ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΑΦΜ`,
    `ΑΔΤ`,
    `ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`,
    `ΦΥΛΟ`,
    `ΕΠΩΝΥΜΟ`,
    `ΟΝΟΜΑ`,
    `ΠΑΤΡΩΝΥΜΟ`,
    `ΕΠΩΝΥΜΙΑ_ΕΤΑΙΡΙΑΣ`,
    `ΔΙΑΚΡΙΤΙΚΟΣ_ΤΙΤΛΟΣ`,
    `ΚΥΚΛΟΣ_ΕΡΓΑΣΙΩΝ_ΕΤΑΙΡΙΑΣ`,
    `ΕΤΟΣ_ΚΥΚΛΟΥ_ΕΡΓΑΣΙΩΝ_ΕΤΑΙΡΙΑΣ`,
    `ΟΔΟΣ`,
    `ΑΡΙΘΜΟΣ`,
    `ΠΟΛΗ`,
    `ΤΑΧ_ΚΩΔΙΚΑΣ`,
    `ΣΥΝΟΛΙΚΟ_ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΚΩΔΙΚΟΣ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΜΕΤΑΒΑΣΗΣ_ΣΤΙΣ_ΜΕΣΑΙΕΣ_Η_ΜΕΓΑΛΕΣ_ΕΠΙΧΕΙΡΗΣΕΙΣ`,
    `ΚΩΔΙΚΟΣ_ΕΠΙΒΕΒΑΙΩΣΗΣ`,
    `ΚΛΑΔΟΣ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`,
    `ΤΑΞΙΝΟΜΗΣΗ_ΚΛΑΔΟΥ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΕΝΗΜΕΡΩΣΗΣ_ΚΛΑΔΟΥ`,
    `ΝΟΜΙΚΗ_ΜΟΡΦΗ_ΕΤΑΙΡΕΙΑΣ`,
    `ΕΙΔΟΣ_HAIRCUT_ΛΟΓΩ_ΡΥΘΜΙΣΗΣ`,
    `ΠΟΣΟΣΤΟ_HAIRCUT_ΛΟΓΩ_ΡΥΘΜΙΣΗΣ`,
    `ΔΙΑΡΚΕΙΑ_ΡΥΘΜΙΣΗΣ`,
    `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`,
    `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`,
    `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,
   `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗ`,
    `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΜΕΓΙΣΤΗ_ΚΑΘΥΣΤΕΡΗΣΗ_ΡΥΘΜ_ΧΟΡΗΓ`,
    `ΔΙΑΡΚΕΙΑ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΠΕΡΙΟΔΟΣ_ΧΑΡΙΤΟΣ_ΤΟΚΩΝ`,
    `ΗΜ_ΡΥΘΜΙΣΗΣ`,
    `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥ`,
    `ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥ`,
    `ΚΩΔΙΚΟΣ_ΝΕΑΣ_ΡΥΘΜΙΣΗΣ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΗΜ/ΝΙΑ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`,
    loanstatus,--exists loan_status for number format from old dwh
    isActive,
    cdi,
    is_open_loan,
    `ΗΜ_ΜΕΤΑΒΟΛΗΣ_ΠΟΣΟΥ__ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `Αριθμός_Εργαζομένων`,
    `ΜΕΓΕΘΟΣ_ΕΠΙΧΕΙΡΗΣΗΣ`,
    `ΌΡΙΟ_ΕΓΚΡΙΣΗΣ_ΑΠΟ_ΙΔΙΑ_ΚΕΦΑΛΑΙΑ_ΤΗΣ_ΤΡΑΠΕΖΑΣ`,
    `ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΑΠΟ_ΙΔΙΑ_ΚΕΦΑΛΑΙΑ_ΤΗΣ_ΤΡΑΠΕΖΑΣ`,
    `ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`,
    `Αριθμός_Σύμβασης_κοινοπρακτικού_δανείου`,
    `Συνολικό_εγκριθέν_ποσό_της_σύμβασης_κοινοπρακτικού_δανείου`,
    `Δείκτης_ποσοστού_συμμετοχής_κοινοπρακτικού_δανείου`,
    `ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ`,
    `ΑΡΙΘΜΟΣ_ΑΠΟΦΑΣΗΣ`,
    `RRE_για_ΑΠΦ`
)
SELECT 
    rpt_date,
    getdate() as created,
    portfolio,
    portfolioid,
    `ΑΥΞΩΝ_ΑΡΙΘΜΟΣ`,
    `ΚΩΔΙΚΟΣ_ΚΑΤΑΣΤΗΜΑΤΟΣ`,
    `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`,
    `ΗΜ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`,
    `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΟΥ_ΟΡΙΟΥ_ΠΟΣΟΥ_ΕΓΚΡΙΣΗΣ_ΜΕ_ΑΛΛΗ_ΕΠΙΧΕΙΡΗΣΗ`,
    `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΕΓΚΡΙΣΗΣ_ΧΟΡΗΓΗΣΕΩΝ_ΤΗΣ_ΙΔΙΑΣ_ΕΠΙΧΕΙΡΗΣΗΣ`,
    `ΑΡ_ΔΟΣΕΩΝ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΥΠΟΛΕΙΠΟΜΕΝΕΣ_ΔΟΣΕΙΣ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΔΕΙΚΤΗΣ_ΚΑΘΥΣΤΕΡΗΣΗΣ`,
    `ΠΟΣΟ_ΔΟΣΗΣ_ΔΑΝΕΙΟΥ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΠΟΣΟ_ΕΠΟΜΕΝΗΣ_ΕΛΑΧΙΣΤΗΣ_ΚΑΤΑΒΟΛΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ`,
    `ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΟΡΙΟ_ΧΟΡΗΓΗΣΗΣ`,
    `ΣΥΧΝΟΤΗΤΑ_ΔΟΣΕΩΝ_Η_ΕΛΑΧΙΣΤΩΝ_ΚΑΤΑΒΟΛΩΝ`,
    `ΠΟΣΟ_ΠΛΗΡΩΜΩΝ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΜΗΝΑ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΚΙΝΗΣΗΣ_ΛΟΓΑΡΙΑΣΜΟΥ`,
    `ΕΝΗΜΕΡΟ_ΥΠΟΛΟΙΠΟ`,
    `ΥΠΟΛΟΙΠΟ_ΣΕ_ΚΑΘΥΣΤΕΡΗΣΗ`,
    `ΑΡΝΗΤΙΚΟ_ΥΠΟΛΟΙΠΟ`,
    `ΣΥΝΟΛΟ_ΟΦΕΙΛΗΣ`,
    `ΗΜ_ΕΓΚΡΙΣΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,
    `ΔΕΙΚΤΗΣ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΑΦΜ`,
    `ΑΔΤ`,
    `ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`,
    `ΦΥΛΟ`,
    `ΕΠΩΝΥΜΟ`,
    `ΟΝΟΜΑ`,
    `ΠΑΤΡΩΝΥΜΟ`,
    `ΕΠΩΝΥΜΙΑ_ΕΤΑΙΡΙΑΣ`,
    `ΔΙΑΚΡΙΤΙΚΟΣ_ΤΙΤΛΟΣ`,
    `ΚΥΚΛΟΣ_ΕΡΓΑΣΙΩΝ_ΕΤΑΙΡΙΑΣ`,
    `ΕΤΟΣ_ΚΥΚΛΟΥ_ΕΡΓΑΣΙΩΝ_ΕΤΑΙΡΙΑΣ`,
    `ΟΔΟΣ`,
    `ΑΡΙΘΜΟΣ`,
    `ΠΟΛΗ`,
    `ΤΑΧ_ΚΩΔΙΚΑΣ`,
    `ΣΥΝΟΛΙΚΟ_ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΚΩΔΙΚΟΣ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΜΕΤΑΒΑΣΗΣ_ΣΤΙΣ_ΜΕΣΑΙΕΣ_Η_ΜΕΓΑΛΕΣ_ΕΠΙΧΕΙΡΗΣΕΙΣ`,
    `ΚΩΔΙΚΟΣ_ΕΠΙΒΕΒΑΙΩΣΗΣ`,
    `ΚΛΑΔΟΣ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`,
    `ΤΑΞΙΝΟΜΗΣΗ_ΚΛΑΔΟΥ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΕΝΗΜΕΡΩΣΗΣ_ΚΛΑΔΟΥ`,
    `ΝΟΜΙΚΗ_ΜΟΡΦΗ_ΕΤΑΙΡΕΙΑΣ`,
    `ΕΙΔΟΣ_HAIRCUT_ΛΟΓΩ_ΡΥΘΜΙΣΗΣ`,
    `ΠΟΣΟΣΤΟ_HAIRCUT_ΛΟΓΩ_ΡΥΘΜΙΣΗΣ`,
    `ΔΙΑΡΚΕΙΑ_ΡΥΘΜΙΣΗΣ`,
    `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`,
    `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`,
    `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΜΕΓΙΣΤΗ_ΚΑΘΥΣΤΕΡΗΣΗ_ΡΥΘΜ_ΧΟΡΗΓ`,
    `ΔΙΑΡΚΕΙΑ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΠΕΡΙΟΔΟΣ_ΧΑΡΙΤΟΣ_ΤΟΚΩΝ`,
    `ΗΜ_ΡΥΘΜΙΣΗΣ`,
    `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΚΩΔΙΚΟΣ_ΝΕΑΣ_ΡΥΘΜΙΣΗΣ_βάσει_ΠΕΕ`,
    `ΗΜ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`,
    loan_status,
    isActive,
    cdi,
    is_open_loan,
    `ΗΜ_ΜΕΤΑΒΟΛΗΣ_ΠΟΣΟΥ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `Αριθμός_Εργαζομένων`,
    `ΜΕΓΕΘΟΣ_ΕΠΙΧΕΙΡΗΣΗΣ`,
    `ΌΡΙΟ_ΕΓΚΡΙΣΗΣ_ΑΠΟ_ΙΔΙΑ_ΚΕΦΑΛΑΙΑ_ΤΗΣ_ΤΡΑΠΕΖΑΣ`,
    `ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΑΠΟ_ΙΔΙΑ_ΚΕΦΑΛΑΙΑ_ΤΗΣ_ΤΡΑΠΕΖΑΣ`,
    `ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`,
    `Αριθμός_Σύμβασης_κοινοπρακτικού_δανείου`,
    `Συνολικό_εγκριθέν_ποσό_της_σύμβασης_κοινοπρακτικού_δανείου`,
    `Δείκτης_ποσοστού_συμμετοχής_κοινοπρακτικού_δανείου`,
    `ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ`,
    `ΑΡΙΘΜΟΣ_ΑΠΟΦΑΣΗΣ`
    ,`RRE_για_ΑΠΦ`
FROM   `{env}-cdh-delivery`.tiresias.EXPORT_BUSINESS
""")

# display(export_business_hist_insert)

## Consumer Table

In [0]:
date1 = spark.sql(f"SELECT DISTINCT rpt_date FROM `{env}-cdh-delivery`.tiresias.export_consumer ").collect()[0][0]

date2 = spark.sql(f"select MAX(rpt_date)  from `{env}-cdh-delivery`.tiresias.export_consumer_hist").collect()[0][0]

# Check if they are equal
if date1 == date2:
    raise ValueError(f"Error: There are already data in consumer_hist for: {date1}. Stopping execution.")
else:
    print(f"Dates are different: {date1} vs {date2}, continuing execution.")

In [0]:
export_consumer_hist_insert = spark.sql(f"""
INSERT INTO `{env}-cdh-delivery`.tiresias.export_consumer_hist (
    rpt_date,
    created,
    portfolio,
    `ΑΥΞΩΝ_ΑΡΙΘΜΟΣ`,
    `ΚΩΔΙΚΟΣ_ΚΑΤΑΣΤΗΜΑΤΟΣ`,
    `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ`,
    `ΗΜ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`,
    `ΑΡ_ΔΟΣΕΩΝ_ΔΑΝΕΙΟΥ`,
    `ΔΕΙΚΤΗΣ_ΚΑΘΥΣΤΕΡΗΣΗΣ`,
    `ΠΟΣΟ_ΔΟΣΗΣ`,
    `ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΔΑΝΕΙΟΥ`,
    `ΣΥΧΝΟΤΗΤΑ_ΔΟΣΕΩΝ`,
    `ΕΝΗΜΕΡΟ_ΥΠΟΛΟΙΠΟ`,
    `ΥΠΟΛΟΙΠΟ_ΣΕ_ΚΑΘΥΣΤΕΡΗΣΗ`,
    `ΑΡΝΗΤΙΚΟ_ΥΠΟΛΟΙΠΟ`,
    `ΣΥΝΟΛΟ_ΟΦΕΙΛΗΣ`,
    `ΗΜ_ΕΓΚΡΙΣΗΣ_ΔΑΝΕΙΟΥΥ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,
    `ΔΕΙΚΤΗΣ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΑΦΜ`,
    `ΑΔΤ`,
    `ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`,
    `ΦΥΛΟ`,
    `ΕΠΩΝΥΜΟ`,
    `ΟΝΟΜΑ`,
    `ΠΑΤΡΩΝΥΜΟ`,
    `ΟΔΟΣ`,
    `ΑΡΙΘΜΟΣ`,
    `ΠΟΛΗ`,
    `ΤΑΧ_ΚΩΔΙΚΑΣ`,
    `ΣΥΝΟΛΙΚΟ_ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΣΤΕΓ_ΔΑΝΕΙΟΥ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΚΩΔΙΚΟΣ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΔΑΝΕΙΟΥ`,
    `ΚΩΔΙΚΟΣ_ΕΠΙΒΕΒΑΙΩΣΗΣ`,
    `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`,
    `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`,
    `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗ`,
    `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΜΕΓΙΣΤΗ_ΚΑΘΥΣΤΕΡΗΣΗ_ΡΥΘΜ_ΧΟΡΗΓ`,
    `ΔΙΑΡΚΕΙΑ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΠΕΡΙΟΔΟΣ_ΧΑΡΙΤΟΣ_ΤΟΚΩΝ`,
    `ΗΜ_ΡΥΘΜΙΣΗΣ`,
    `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥ`,
    `ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥ`,
    `ΚΩΔΙΚΟΣ_ΝΕΑΣ_ΡΥΘΜΙΣΗΣ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΗΜ/ΝΙΑ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`,
    `ΚΑΝΑΛΙ_ΔΙΑΘΕΣΗΣ_ΧΟΡΗΓΗΣΗΣ`,
    loanstatus,--exists loan_status for number format from old dwh
    isActive,
    cdi,
    is_open_loan,
    `ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ`,
    `ΑΡΙΘΜΟΣ_ΑΠΟΦΑΣΗΣ`,
    `RRE_για_ΑΠΦ`
)
SELECT
    rpt_date,
    getdate() as created,
    portfolio,
    `ΑΥΞΩΝ_ΑΡΙΘΜΟΣ`,
    `ΚΩΔΙΚΟΣ_ΚΑΤΑΣΤΗΜΑΤΟΣ`,
    `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
    `ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΗΜ_ΚΛΕΙΣΙΜΑΤΟΣ`,
    `ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`,
    `ΗΜ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ_OVERDRAFT`,
    `ΑΡ_ΔΟΣΕΩΝ_ΔΑΝΕΙΟΥ`,
    `ΔΕΙΚΤΗΣ_ΚΑΘΥΣΤΕΡΗΣΗΣ`,
    `ΠΟΣΟ_ΔΟΣΗΣ`,
    `ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΔΑΝΕΙΟΥ`,
    `ΣΥΧΝΟΤΗΤΑ_ΔΟΣΕΩΝ`,
    `ΕΝΗΜΕΡΟ_ΥΠΟΛΟΙΠΟ`,
    `ΥΠΟΛΟΙΠΟ_ΣΕ_ΚΑΘΥΣΤΕΡΗΣΗ`,
    `ΑΡΝΗΤΙΚΟ_ΥΠΟΛΟΙΠΟ`,
    `ΣΥΝΟΛΟ_ΟΦΕΙΛΗΣ`,
    `ΗΜ_ΕΓΚΡΙΣΗΣ_ΔΑΝΕΙΟΥ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`,
    `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,
    `ΔΕΙΚΤΗΣ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΗΜ_ΜΗ_ΜΕΤΑΔΟΣΗΣ`,
    `ΑΦΜ`,
    `ΑΔΤ`,
    `ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`,
    `ΦΥΛΟ`,
    `ΕΠΩΝΥΜΟ`,
    `ΟΝΟΜΑ`,
    `ΠΑΤΡΩΝΥΜΟ`,
    `ΟΔΟΣ`,
    `ΑΡΙΘΜΟΣ`,
    `ΠΟΛΗ`,
    `ΤΑΧ_ΚΩΔΙΚΑΣ`,
    `ΣΥΝΟΛΙΚΟ_ΠΟΣΟ_ΕΚΤΑΜΙΕΥΣΗΣ_ΣΤΕΓ_ΔΑΝΕΙΟΥ`,
    `ΗΜΕΡΟΜΗΝΙΑ_ΤΕΛΕΥΤΑΙΑΣ_ΕΚΤΑΜΙΕΥΣΗΣ_ΓΙΑ_ΧΟΡΗΓΗΣΕΙΣ_ΤΑΚΤΗΣ_ΛΗΞΗΣ`,
    `ΚΩΔΙΚΟΣ_ΠΡΟΗΓΟΥΜΕΝΟΥ_ΔΑΝΕΙΟΥ`,
    `ΚΩΔΙΚΟΣ_ΕΠΙΒΕΒΑΙΩΣΗΣ`,
    `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`,
    `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`,
    `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`,
    `ΔΕΙΚΤΗΣ_ΡΥΘΜΙΣΗΣ`,
    `ΜΕΓΙΣΤΗ_ΚΑΘΥΣΤΕΡΗΣΗ_ΡΥΘΜ_ΧΟΡΗΓ`,
    `ΔΙΑΡΚΕΙΑ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΗΜ_ΛΗΞΗΣ_ΠΕΡΙΟΔΟΥ_ΧΑΡΙΤΟΣ`,
    `ΠΕΡΙΟΔΟΣ_ΧΑΡΙΤΟΣ_ΤΟΚΩΝ`,
    `ΗΜ_ΡΥΘΜΙΣΗΣ`,
    `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΗΜ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`,
    `ΚΩΔΙΚΟΣ_ΝΕΑΣ_ΡΥΘΜΙΣΗΣ_ΒΑΣΕΙ_ΠΕΕ`,
    `ΗΜ/ΝΙΑ_ΑΝΑΦΟΡΑΣ_ΡΥΘΜΙΣΗΣ_ΩΣ_ΠΕΕ`,
    `ΚΑΝΑΛΙ_ΔΙΑΘΕΣΗΣ_ΧΟΡΗΓΗΣΗΣ`,
    loan_status,
    isActive,
    cdi,
    is_open_loan,
    `ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ`,
    `ΑΡΙΘΜΟΣ_ΑΠΟΦΑΣΗΣ`,
    `RRE_για_ΑΠΦ`
FROM `{env}-cdh-delivery`.tiresias.export_consumer
""")

# display(export_consumer_hist_insert)